# Bài học 02 - Khám phá Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** là một framework hợp nhất để xây dựng các AI Agent. Nó cung cấp một kiến trúc rõ ràng, có khả năng kết hợp với bốn khối xây dựng cốt lõi:

- **Client** – kết nối đến một endpoint mô hình AI và xử lý giao tiếp
- **Agent** – bọc một client với các chỉ thị (instructions) và định nghĩa công cụ (tools)
- **Tools** – mở rộng khả năng của agent bằng các hàm tùy chỉnh mà mô hình có thể gọi
- **Session** – duy trì lịch sử hội thoại cho các tương tác nhiều lượt (multi-turn)

Trong bài học này, chúng ta sẽ xây dựng một **agent đặt vé du lịch** (travel booking agent) có khả năng kiểm tra tình trạng trống của điểm đến bằng cách sử dụng các khái niệm trên.

## Thiết lập (Setup)

In [ ]:
# Cài đặt package Microsoft Agent Framework
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Hiểu về Kiến trúc Agent Framework

Microsoft Agent Framework tuân theo kiến trúc phân lớp:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `AzureAIProjectAgentProvider` kết nối đến một deployment Azure OpenAI. Nó xử lý xác thực, định dạng yêu cầu, và phân tích phản hồi.
2. **Agent** – Được tạo từ client qua `provider.create_agent()`, agent kết hợp quyền truy cập mô hình với các chỉ thị (system prompt) và các công cụ.
3. **Tools** – Các hàm Python được gắn decorator `@tool` mà agent có thể gọi để thực hiện hành động hoặc truy xuất dữ liệu.
4. **Session** – Đối tượng `AgentSession` (được tạo qua `agent.create_session()`) lưu trữ lịch sử hội thoại, cho phép đối thoại nhiều lượt mà agent có thể nhớ ngữ cảnh trước đó.

Hãy cùng xây dựng từng lớp một.

In [ ]:
# Tạo client – đây là kết nối tới mô hình AI
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Thêm các Công cụ (Tools) với Decorator @tool

Các công cụ (tools) cho phép agent thực hiện các hành động vượt xa việc chỉ tạo ra văn bản. Decorator `@tool` chuyển đổi một hàm Python thông thường thành một thứ mà agent có thể gọi.

Những điểm chính:
- Sử dụng `Annotated[type, "description"]` để mô hình hiểu rõ từng tham số.
- Docstring (chuỗi tài liệu) trở thành mô tả công cụ mà mô hình sẽ đọc.
- `approval_mode="never_require"` có nghĩa là công cụ tự động chạy mà không cần người dùng xác nhận.

In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Tạo Agent với các Công cụ

Bây giờ chúng ta kết hợp client, chỉ thị và các công cụ vào một agent. `instructions` đóng vai trò là system prompt — định nghĩa tính cách và hành vi của agent.

In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "Bạn là một agent đặt vé du lịch. Hãy giúp người dùng kiểm tra xem điểm đến có trống hay không "
        "và đưa ra các đề xuất. Luôn kiểm tra tính khả dụng trước khi đề xuất một điểm đến."
    ),
    tools=[check_destination_availability],
)

## Hội thoại Nhiều lượt (Multi-Turn) với Session

Một `AgentSession` (được tạo qua `agent.create_session()`) theo dõi mọi tin nhắn trong một cuộc hội thoại. Bằng cách truyền cùng một session vào mỗi lần gọi `agent.run()`, agent có quyền truy cập toàn bộ lịch sử hội thoại và có thể tham chiếu lại những tin nhắn trước đó.

Chúng ta truyền `tools=[check_destination_availability]` để agent có thể gọi bộ kiểm tra tính khả dụng của chúng ta trong mỗi lượt.

In [ ]:
session = agent.create_session()

# Lượt 1: Hỏi về các điểm đến đang trống
response = await agent.run(
    "Bạn có những điểm đến nào đang trống?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Lượt 2: Câu hỏi theo dõi — agent sẽ nhớ lại cuộc trò chuyện
response = await agent.run(
    "Tôi muốn đi đâu đó thật ấm áp. Có nơi nào đang trống không?",
    session=session,
)
print(f"Agent: {response}")

## Tóm tắt

Trong bài học này, bạn đã khám phá bốn trụ cột của Microsoft Agent Framework:

| Khái niệm | Điều bạn đã học |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` kết nối tới Azure OpenAI với xác thực bằng thông tin đăng nhập |
| **Agent** | `provider.create_agent()` kết hợp kết nối mô hình với các chỉ thị và tên gọi |
| **Tools** | Decorator `@tool` phơi bày các hàm Python để agent có thể gọi |
| **Session** | `agent.create_session()` duy trì lịch sử cuộc trò chuyện xuyên suốt nhiều lượt |

Các khối xây dựng này kết hợp với nhau để tạo ra những agent có khả năng duy trì các cuộc trò chuyện tự nhiên, gọi các hàm bên ngoài và duy trì ngữ cảnh — nền tảng cho các mẫu agentic nâng cao hơn trong những bài học sau.